In [1]:
from langchain.prompts import PromptTemplate
import os
import json
import openai
from pathlib import Path
import re
import numpy as np

In [ ]:
with open('path to adv outputs') as f:
    data = json.load(f)

In [3]:
with open('issues_examples.json') as f:
    issues_exp = json.load(f)

In [ ]:
data[0]

In [ ]:
issues_exp[data[0]['issue']]

In [6]:
def load_prompt(file_name):
    base_dir = "prompts/"
    file_path = base_dir + file_name
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()

In [ ]:
openai.api_key = "openai-api-key"

In [8]:
prompt_text = load_prompt("evaluation.txt")

In [9]:
prompt_template = PromptTemplate(input_variables=["issue","prev_user_query", "prev_response", "prev_comment_1", "prev_comment_2", "new_user_query", "new_response", ""],
                                    template=prompt_text)

In [10]:
response_data = {}

In [11]:
for x in issues_exp.keys():
    response_data[x] = []

In [12]:
for i in range(len(data)):

    prompt = prompt_template.format(issue=data[i]['issue'],
                                    prev_user_query=issues_exp[data[i]['issue']]['question'],
                                   prev_response=issues_exp[data[i]['issue']]['response'],
                                   prev_comment_1=issues_exp[data[i]['issue']]['comment_1'],
                                   prev_comment_2=issues_exp[data[i]['issue']]['comment_2'],
                                   new_user_query=data[i]['question'],
                                   new_response=data[i]['response'])

    messages = [{'role': 'user', 'content': prompt}]
    response = openai.chat.completions.create(
                            model="gpt-4o",
                            messages=messages,
                            temperature=0,
                            n=1
                            )
    response_raw = response.choices[0].message.content
    response_cleaned = re.sub(r"^```json\s*|\s*```$", "", response_raw.strip())
    response_json = json.loads(response_cleaned)
    response_data[data[i]['issue']].append(response_json)

In [ ]:
for issue in response_data.keys():
    print(issue)
    yes_count = 0
    no_count = 0
    unsure_count = 0
    for response_data_point in response_data[issue]:
        if response_data_point['Answer']=='Yes':
            yes_count = yes_count + 1
        elif response_data_point['Answer']=='No':
            no_count = no_count+1
        else:
            unsure_count = unsure_count+1
    total_sure = len(response_data[issue]) - unsure_count
    print(yes_count/total_sure)

In [ ]:
with open('results file','w') as f:
    json.dump(response_data,f)